# 地方競馬 〜 ② モデルの作り方を変える検証（特徴量は今のまま）3年 × 乱数10回・保存はしない

| 案 | 中身 |
|---|---|
| 今のモデル | 基準（1頭ずつ「勝つか」を学習 → 市場と合成） |
| C1 市場を出発点 | 市場の確率を出発点にして、モデルには「市場とのずれ」だけを学習させる（残差学習） |
| C2 レース単位 | 1レースの中で「どの馬が勝つか」を直接学習する（1レースで勝つのは1頭、をモデルが知っている） |
| C1＋C2 | 市場を出発点にしたレース単位の学習 |
| C7 新しい年を重く | 1年古くなるごとに学習での重みを0.85倍にする（市場が年々賢くなる影響への対応） |
| C5 控えめ設定 | 木を小さく（葉31）・1枝あたりの最低頭数300・使う特徴量6割・正則化あり |
| C5 大きめ設定 | 木を大きく（葉127）・1枝あたりの最低頭数50 |
| C6 CatBoost と平均 | 別の種類のモデル（CatBoost）を学習し、今のモデルと半々で平均 |

後半では、学習し直さずに予想の確率だけを直す方法も比べる（今のモデルの予想を使うので数分）。

| 案 | 中身 |
|---|---|
| C3 オッズ帯の補正 | オッズの帯ごとに「実際の勝ち数 ÷ 予想」で確率を直す（大穴の過小評価への対応） |
| C4 条件ごとの合成の重み | 競馬場・頭数・馬場・レース番号ごとに、モデルと市場の合成の重みを別々に決める |
| D2 見送る条件 | 本番の単勝（◎・期待値1.05以上）の回収率を、競馬場・頭数・レース番号・馬場・距離で分ける |

**判定**: 市場との差で30回中24回以上勝てば効果あり。回収率は本番の基準と、広く振ったマスで確認。

**Input**: 学習用と同じ3つ。Internet は不要。
**実行**: Save & Run All (Commit)。**8〜10時間**の見込み（CatBoost が重い）。10時間を超えそうなら後ろの案（CatBoost → 大きめ → 控えめ …）から自動で省略して結果を出す。

## ⓪ 設定

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

DATA_ROOT   = "/kaggle/input"
TEST_YEARS  = list(range(2016, 2026))  # 1年ずつ評価する年（各年とも「前年まで」のデータだけで学習）
BASE_END    = "2015-01-01"   # スピード指数の基準タイムはこの日より前のデータだけで作る（全ての評価年より過去）
USE_ODDS    = True          # True: オッズも特徴量に入れ、「市場の評価のどこがズレているか」を直接学習させる
import time
T_NB = time.time()   # 時間切れ（Kaggleは12時間で強制終了）を防ぐための開始時刻
TIME_LIMIT_H = 10.0   # これを超えたら残りの案を省略して結果を出す

## ① 読込（オッズ "2,012.0" のカンマで割れた行を直して読む）

In [ ]:
DIR = os.path.dirname(glob.glob(f"{DATA_ROOT}/**/keiba_results.csv", recursive=True)[0])

def fix_row(row):
    if len(row) == 19:
        return row[:15] + [row[15] + row[16]] + row[17:]
    return None

res = pd.read_csv(f"{DIR}/keiba_results.csv", engine="python", on_bad_lines=fix_row,
                  dtype={"race_id": str, "horse_id": str, "jockey_id": str})
races = pd.read_csv(f"{DIR}/keiba_races.csv", dtype={"race_id": str})
pays  = pd.read_csv(f"{DIR}/keiba_payouts.csv", dtype={"race_id": str, "horse_num": str})
# 追加データ（2026年収集分・過去の穴埋め分）。収集ノートブックの出力を Private データセットにして Add Input しておくと自動で結合
for tag in ["2026", "fill"]:
    add = {n: sorted(glob.glob(f"{DATA_ROOT}/**/keiba_{n}_{tag}.csv", recursive=True)) for n in ["results", "races", "payouts"]}
    if not all(add.values()):
        continue
    res = pd.concat([res] + [pd.read_csv(p, dtype={"race_id": str, "horse_id": str, "jockey_id": str}) for p in add["results"]],
                    ignore_index=True).drop_duplicates(["race_id", "number"], keep="last")
    races = pd.concat([races] + [pd.read_csv(p, dtype={"race_id": str}) for p in add["races"]],
                      ignore_index=True).drop_duplicates("race_id", keep="last")
    pays = pd.concat([pays] + [pd.read_csv(p, dtype={"race_id": str, "horse_num": str}) for p in add["payouts"]],
                     ignore_index=True).drop_duplicates(keep="last")
    print(f"追加データ（{tag}）を結合しました:", add["races"])
print("results:", res.shape, "/ races:", races.shape, "/ payouts:", pays.shape)

## ② 地方に絞り、race_id から日付を作る

地方の race_id は「年(4) + 場(2) + 月(2) + 日(2) + R(2)」。帯広（ばんえい, 場コード65）は別競技なので除外。
下の確認で「月」が 1〜12 だけ、「日」が 1〜31 だけになっていれば解釈は正しい。

In [ ]:
EXCLUDE_VENUES = {"65"}

def is_nar(rid):
    code = rid.str[4:6]
    return ~code.astype(int).between(1, 10) & ~code.isin(EXCLUDE_VENUES)

res   = res[is_nar(res["race_id"])].copy()
races = races[is_nar(races["race_id"])].copy()

mm = races["race_id"].str[6:8].astype(int)
dd = races["race_id"].str[8:10].astype(int)
print("月の範囲:", mm.min(), "〜", mm.max(), "/ 日の範囲:", dd.min(), "〜", dd.max())
print("月ごとのレース数:", mm.value_counts().sort_index().to_dict())
assert mm.between(1, 12).all() and dd.between(1, 31).all(), "月日として解釈できない値があります"

races["date"] = pd.to_datetime(races["race_id"].str[:4] + races["race_id"].str[6:10],
                               format="%Y%m%d", errors="coerce")
print("日付が作れなかったレース:", races["date"].isna().sum())
races = races.dropna(subset=["date"])
pays  = pays[pays["race_id"].isin(set(races["race_id"]))].copy()
print("地方レース数:", len(races), "/ 期間:", races["date"].min().date(), "〜", races["date"].max().date())

## ③ 整形

In [ ]:
race_cols = ["race_id", "date", "venue", "race_number", "course_type", "distance",
             "turn", "weather", "track_condition"]
df = res.merge(races[race_cols], on="race_id", how="inner")

# 着順: 取消・除外は出走していないので除く / 中止・失格は最下位扱い / "2(降)" は数字部分
df = df[~df["rank"].astype(str).isin(["取", "除"])].copy()
df["n_runners"] = df.groupby("race_id")["race_id"].transform("size")
df["finish"] = pd.to_numeric(df["rank"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce")
df["finish"] = df["finish"].fillna(df["n_runners"]).astype(int)
df["win"]  = (df["finish"] == 1).astype(int)
df["top3"] = (df["finish"] <= 3).astype(int)
df["finish_ratio"] = (df["finish"] - 1) / (df["n_runners"] - 1).clip(lower=1)

# 数値化
for c in ["frame", "number", "weight", "distance", "race_number", "odds", "popularity", "last_3f"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["sex"] = df["sex_age"].astype(str).str[0]
df["age"] = pd.to_numeric(df["sex_age"].astype(str).str[1:], errors="coerce")
bw = df["horse_weight"].astype(str).str.extract(r"(\d+)\(([-+]?\d+)\)")
df["body_weight"] = pd.to_numeric(bw[0], errors="coerce")
df["bw_diff"]     = pd.to_numeric(bw[1], errors="coerce")
t = df["time"].astype(str).str.extract(r"(?:(\d+):)?(\d+\.\d)")
df["time_sec"] = pd.to_numeric(t[0], errors="coerce").fillna(0) * 60 + pd.to_numeric(t[1], errors="coerce")
pos = df["passing"].astype(str).str.split("-")
df["first_corner"] = pd.to_numeric(pos.str[0], errors="coerce")
df["horse_key"]  = df["horse_id"].fillna(df["horse_name"])
df["jockey_key"] = df["jockey_id"].fillna(df["jockey_name"])
df["month"] = df["date"].dt.month

# レース後にしか分からない「そのレースでの内容」（前走特徴量の材料。そのレースの特徴量には使わない）
df["time_behind"]  = df["time_sec"] - df.groupby("race_id")["time_sec"].transform("min")
df["last3f_rank"]  = df.groupby("race_id")["last_3f"].rank(pct=True)
df["corner1_ratio"] = df["first_corner"] / df["n_runners"]
print(df.shape)

## ④ 過去走の特徴量（必ず shift(1) で当該レースを除外）

スピード指数は「直近3年の基準タイム」と「その日の馬場差」で計算（2026/10 改良）。

F4（メンバーの中での位置: 5走平均・最高指数のメンバー内の偏差、最上位との差、メンバー上位3頭の平均との差）も追加（2026/11）。

In [ ]:
# --- スピード指数: 競馬場・コース・距離・馬場ごとの基準タイムとの差（50が平均、高いほど速い）---
# 基準タイムは BASE_END より前の5着以内の馬だけで作る（どの評価年から見ても過去のデータ）
BASE_KEYS = ["venue", "course_type", "distance", "track_condition"]
base = (df[(df["date"] < BASE_END) & (df["finish"] <= 5)]
        .groupby(BASE_KEYS)["time_sec"].agg(base_mean="mean", base_std="std", base_n="size").reset_index())
base = base[base["base_n"] >= 30]
df = df.drop(columns=["base_mean", "base_std", "base_n", "speed"], errors="ignore").merge(base, on=BASE_KEYS, how="left")
df["speed"] = (df["base_mean"] - df["time_sec"]) / df["base_std"] * 10 + 50
print("スピード指数が計算できた割合:", round(df["speed"].notna().mean(), 3))

# --- スピード指数の作り直し（2026/10 改良：比較実験で5回全勝）---
# 1) 基準タイムを「各年について、その前の3年」で作り直す（馬場の改修や新しい距離設定に追随。未来の年は使わない）
# 2) その日・その競馬場・コースの時計の速さ（馬場差）で補正する（レース後に分かる値で、次走以降の特徴量にだけ使う）
df["year"] = df["date"].dt.year
df["_bk"] = (df["venue"].astype(str) + "|" + df["course_type"].astype(str) + "|" + df["distance"].astype(str)
             + "|" + df["track_condition"].astype(str))
parts = []
for Y in sorted(df["year"].unique()):
    s_ = df[(df["year"] >= Y - 3) & (df["year"] <= Y - 1) & (df["finish"] <= 5)]
    st = s_.groupby("_bk")["time_sec"].agg(b2_mean="mean", b2_std="std", b2_n="size").reset_index()
    st = st[st["b2_n"] >= 30]
    st["year"] = Y
    parts.append(st)
df = df.drop(columns=["b2_mean", "b2_std", "b2_n"], errors="ignore").merge(pd.concat(parts, ignore_index=True), on=["_bk", "year"], how="left")
raw2 = ((df["b2_mean"] - df["time_sec"]) / df["b2_std"] * 10 + 50).fillna(df["speed"])
race_dev = raw2.where(df["finish"] <= 5).groupby(df["race_id"]).transform("mean") - 50
day = df["date"].astype(str) + "_" + df["venue"].astype(str) + "_" + df["course_type"].astype(str)
first = ~df["race_id"].duplicated()
day_dev = race_dev[first].groupby(day[first]).mean()
n_day = first.groupby(day).sum()
variant = day.map(day_dev).where(day.map(n_day) >= 3)
df["speed_old"] = df["speed"]
df["speed"] = raw2 - variant.fillna(0)
print("新しいスピード指数: 直近3年の基準が使えた割合", round(df["b2_mean"].notna().mean(), 3),
      "/ 馬場差で補正できた割合", round(variant.notna().mean(), 3), "/ 計算できた割合", round(df["speed"].notna().mean(), 3))

# 予想用ノートブックで結果を手入力したときに使う基準タイム（直近3年。馬場差の補正は無し）
LY = int(df["year"].max())
base = (df[(df["year"] >= LY - 2) & (df["finish"] <= 5)]
        .groupby(BASE_KEYS, observed=True)["time_sec"].agg(base_mean="mean", base_std="std", base_n="size").reset_index())
base = base[base["base_n"] >= 30]

df = df.sort_values(["date", "race_id", "number"]).reset_index(drop=True)
g = df.groupby("horse_key", sort=False)

def past_mean(col, n):
    s = g[col].shift(1)
    return s.groupby(df["horse_key"]).rolling(n, min_periods=1).mean().reset_index(level=0, drop=True)

for k in (1, 2, 3):
    df[f"prev_finish_{k}"] = g["finish"].shift(k)
df["prev_finish_ratio"]   = g["finish_ratio"].shift(1)
df["prev_speed"]          = g["speed"].shift(1)
df["avg_speed_5"]         = past_mean("speed", 5)
df["max_speed_5"]         = (g["speed"].shift(1).groupby(df["horse_key"]).rolling(5, min_periods=1).max()
                             .reset_index(level=0, drop=True))
df["avg_finish_ratio_5"]  = past_mean("finish_ratio", 5)
df["top3_rate_5"]         = past_mean("top3", 5)
df["win_rate_10"]         = past_mean("win", 10)
df["prev_time_behind"]    = g["time_behind"].shift(1)
df["avg_time_behind_5"]   = past_mean("time_behind", 5)
df["prev_last3f_rank"]    = g["last3f_rank"].shift(1)
df["avg_last3f_rank_5"]   = past_mean("last3f_rank", 5)
df["prev_corner1_ratio"]  = g["corner1_ratio"].shift(1)
df["avg_corner1_ratio_5"] = past_mean("corner1_ratio", 5)
df["prev_popularity"]     = g["popularity"].shift(1)
df["prev_log_odds"]       = np.log1p(g["odds"].shift(1))
df["n_races"]             = g.cumcount()
df["days_since_last"]     = (df["date"] - g["date"].shift(1)).dt.days
df["dist_diff"]           = df["distance"] - g["distance"].shift(1)
df["weight_diff"]         = df["weight"] - g["weight"].shift(1)
df["same_venue_as_prev"]  = (df["venue"] == g["venue"].shift(1)).astype(float)

# 騎手・調教師の通算成績（前日までの成績のみ。平滑化つき）
K = 30
def past_rate(key, target):
    daily = (df.groupby([key, "date"])[target].agg(["sum", "count"])
               .reset_index().sort_values([key, "date"]))
    gg = daily.groupby(key)
    past_sum = gg["sum"].cumsum() - daily["sum"]
    past_cnt = gg["count"].cumsum() - daily["count"]
    p = df[target].mean()
    daily[f"{key}_{target}_rate"] = (past_sum + p * K) / (past_cnt + K)
    return daily[[key, "date", f"{key}_{target}_rate"]]

for key, target in [("jockey_key", "top3"), ("jockey_key", "win"), ("trainer", "top3")]:
    df = df.merge(past_rate(key, target), on=[key, "date"], how="left")

# レース内での相対位置（メンバーの中で強いか）
for c in ["avg_finish_ratio_5", "top3_rate_5", "avg_time_behind_5", "jockey_key_top3_rate",
          "avg_speed_5", "max_speed_5"]:
    df[f"{c}_inrace"] = df.groupby("race_id")[c].rank(pct=True)

# F4 メンバーの中での位置（2026/11 改良：3年×乱数10回の検証で26勝4敗）
for c in ["avg_speed_5", "max_speed_5"]:
    gc = df.groupby("race_id")[c]
    df[f"{c}_z"] = (df[c] - gc.transform("mean")) / gc.transform("std").replace(0, np.nan)
df["max_speed_5_gap"] = df["max_speed_5"] - df.groupby("race_id")["max_speed_5"].transform("max")
rk = df.groupby("race_id")["max_speed_5"].rank(ascending=False, method="first")
df["field_level"] = df["max_speed_5"].where(rk <= 3).groupby(df["race_id"]).transform("mean")
df["max_speed_5_vs_field"] = df["max_speed_5"] - df["field_level"]
F4_COLS = ["avg_speed_5_z", "max_speed_5_z", "max_speed_5_gap", "field_level", "max_speed_5_vs_field"]

cat_cols = ["venue", "course_type", "turn", "weather", "track_condition", "sex"]
for c in cat_cols:
    df[c] = df[c].astype("category")

features = (
    ["frame", "number", "weight", "age", "body_weight", "bw_diff", "distance", "race_number",
     "n_runners", "month"] + cat_cols +
    ["prev_finish_1", "prev_finish_2", "prev_finish_3", "prev_finish_ratio", "avg_finish_ratio_5",
     "prev_speed", "avg_speed_5", "max_speed_5",
     "top3_rate_5", "win_rate_10", "prev_time_behind", "avg_time_behind_5", "prev_last3f_rank",
     "avg_last3f_rank_5", "prev_corner1_ratio", "avg_corner1_ratio_5", "prev_popularity",
     "prev_log_odds", "n_races", "days_since_last", "dist_diff", "weight_diff", "same_venue_as_prev",
     "jockey_key_top3_rate", "jockey_key_win_rate", "trainer_top3_rate"] +
    [c for c in df.columns if c.endswith("_inrace")] + F4_COLS
)
if USE_ODDS:
    features.append("odds")
print(len(features), "features")

## ⑤ 比較の部品

In [ ]:
from scipy.optimize import minimize
import re
from itertools import combinations

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
df["year"] = df["date"].dt.year
inv = 1 / df["odds"].where(df["odds"] > 0)
df["p_mkt"] = inv / inv.groupby(df["race_id"]).transform("sum")
df["race_ok"] = (df.groupby("race_id")["p_mkt"].transform(lambda s: s.notna().all())
                 & (df.groupby("race_id")["win"].transform("sum") == 1))
df["rel"] = np.clip(4 - df["finish"].fillna(99), 0, 3).astype(int)     # レース単位の学習用: 1着3・2着2・3着1・他0
LAST = int(df["year"].max())
YEARS = [LAST, LAST - 1, LAST - 2]
order = ["date", "race_id", "number"]
base_params = dict(n_estimators=3000, learning_rate=0.05, num_leaves=63, min_child_samples=100,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)

def race_norm(x, rid):
    x = pd.Series(np.asarray(x, dtype=float), index=rid.index)
    return (x / x.groupby(rid).transform("sum")).values

def race_softmax(score, rid):
    s = pd.Series(np.asarray(score, dtype=float), index=rid.index)
    e = np.exp(s - s.groupby(rid).transform("max"))
    return (e / e.groupby(rid).transform("sum")).values

def comb(d, w):
    s = w[0] * np.log(d["pm"].clip(1e-6)) + w[1] * np.log(d["p_mkt"].clip(1e-6))
    e = np.exp(s - s.groupby(d["race_id"]).transform("max"))
    return e / e.groupby(d["race_id"]).transform("sum")

def ll(w, d):
    return -np.log(comb(d, w)[d["win"] == 1].clip(1e-9)).mean()

def gap(d_fit, d_ev):
    w = minimize(ll, x0=[0.5, 0.5], args=(d_fit,), method="Nelder-Mead").x
    return ll([0, 1], d_ev) - ll(w, d_ev), w

def set_period(Y):
    global tr, va_fit, va_ev
    tr = df[df["year"] <= Y - 1].sort_values(order)
    v_ = df[(df["year"] == Y) & df["race_ok"]].sort_values(order)
    mid_ = v_["date"].drop_duplicates().sort_values().iloc[v_["date"].nunique() // 2]
    va_fit, va_ev = v_[v_["date"] < mid_], v_[v_["date"] >= mid_]
    print(f"\n===== {Y}年: 学習 〜{Y - 1}年 {tr['race_id'].nunique():,}レース / 前半 {va_fit['race_id'].nunique():,} / 後半（採点）{va_ev['race_id'].nunique():,}（{mid_.date()}〜） =====")

def groups(d):
    return d.groupby("race_id", sort=False).size().values

def fit_predict(feats, mode, sd):
    if mode == "rank":
        m = lgb.LGBMRanker(**{**base_params, "random_state": sd, "objective": "lambdarank"})
        m.fit(tr[feats], tr["rel"], group=groups(tr), eval_set=[(va_fit[feats], va_fit["rel"])],
              eval_group=[groups(va_fit)], eval_at=[1], callbacks=[lgb.early_stopping(100, verbose=False)])
        pf = race_softmax(m.predict(va_fit[feats]), va_fit["race_id"])
        pe = race_softmax(m.predict(va_ev[feats]), va_ev["race_id"])
    else:
        m = lgb.LGBMClassifier(**{**base_params, "random_state": sd})
        m.fit(tr[feats], tr["win"], eval_set=[(va_fit[feats], va_fit["win"])], eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        pf = race_norm(m.predict_proba(va_fit[feats])[:, 1], va_fit["race_id"])
        pe = race_norm(m.predict_proba(va_ev[feats])[:, 1], va_ev["race_id"])
    return pf, pe, m.best_iteration_, m

COLS = ["race_id", "date", "number", "win", "p_mkt", "odds", "n_runners"]
def run(name, feats, mode):
    t0 = time.time()
    pfs, pes, its, sg = [], [], [], []
    for sd in SEEDS:
        pf, pe, it, m = fit_predict(feats, mode, sd)
        pfs.append(pf); pes.append(pe); its.append(it)
        sg.append(gap(va_fit[COLS].assign(pm=pf), va_ev[COLS].assign(pm=pe))[0])
    pf, pe = np.mean(pfs, axis=0), np.mean(pes, axis=0)
    g1, w1 = gap(va_fit[COLS].assign(pm=pf), va_ev[COLS].assign(pm=pe))
    E = va_ev[COLS].assign(pm=pe)
    E["p"] = comb(E, w1).values
    E["ev"] = E["p"] * E["odds"]
    r = dict(市場との差=g1, 木=int(np.mean(its)), 秒=int(time.time() - t0), _seed=sg, _w=w1, _E=E,
             _imp=dict(zip(feats, m.feature_importances_)))
    print(f"{name}: 市場との差 {g1:.5f}（乱数ごと " + " / ".join(f"{x:.5f}" for x in sg)
          + f"）/ 合成の重み モデル {w1[0]:.3f}・市場 {w1[1]:.3f} / 木 {r['木']} / {r['秒']}秒")
    return r

# ---------- 回収率（案F の単勝・度合い1.6以上のワイド）----------
wp = pays[pays["bet_type"] == "ワイド"].copy()
def _pk(s):
    nums = sorted(int(x) for x in re.findall(r"\d+", str(s)))
    return f"{nums[0]}-{nums[1]}" if len(nums) == 2 else None
wp["key"] = wp["horse_num"].map(_pk)
wp["payout"] = pd.to_numeric(wp["payout"].astype(str).str.replace(",", ""), errors="coerce")
WPAY = wp.dropna(subset=["key", "payout"]).groupby(["race_id", "key"])["payout"].max().to_dict()
HAVE_WIDE = set(wp["race_id"])
N_SIM = 2000
GUM, MKT_MAT = {}, {}

def pair_mat(p, g, k):
    s = g + np.log(np.clip(p, 1e-9, None))
    top = np.argpartition(-s, k - 1, axis=1)[:, :k]
    hit = np.zeros(s.shape, dtype=np.float32)
    np.put_along_axis(hit, top, 1.0, axis=1)
    return (hit.T @ hit) / len(s)

def roi(E):
    s = E[(E["ev"] >= 1.2) & (E["p"] >= 0.10)]
    s = s.loc[s.groupby("race_id")["ev"].idxmax()] if len(s) else s
    tan = (s["odds"] * s["win"]).values
    wide = []
    for rid, X in E[E["race_id"].isin(HAVE_WIDE) & (E["n_runners"] >= 5)].groupby("race_id", sort=False):
        n = len(X); k = 3 if n >= 8 else 2
        if rid not in GUM:
            GUM[rid] = np.random.default_rng(abs(hash(rid)) % 2**32).gumbel(size=(N_SIM, n))
            MKT_MAT[rid] = pair_mat(X["p_mkt"].values, GUM[rid], k)
        mm = pair_mat(X["p"].values, GUM[rid], k)
        iu = np.triu_indices(n, 1)
        pm, pk = mm[iu], MKT_MAT[rid][iu]
        ok = (pm >= 0.2) & (pm / np.maximum(pk, 1e-9) >= 1.6)
        if ok.any():
            j = np.argmax(np.where(ok, pm / np.maximum(pk, 1e-9), -1))
            nums = X["number"].astype(int).values
            a, b = sorted((nums[iu[0][j]], nums[iu[1][j]]))
            wide.append(WPAY.get((rid, f"{a}-{b}"), 0) / 100)
    return tan, np.array(wide)

In [ ]:
# ---------- 買い基準を広く振るための材料を集める（モデルごと・年ごと）----------
def collect(E, Y):
    T_ = E[["race_id", "date", "p", "ev", "odds", "win"]].assign(年=Y)
    rows = []
    for rid, X in E[E["race_id"].isin(HAVE_WIDE) & (E["n_runners"] >= 5)].groupby("race_id", sort=False):
        n = len(X); k = 3 if n >= 8 else 2
        if rid not in GUM:
            GUM[rid] = np.random.default_rng(abs(hash(rid)) % 2**32).gumbel(size=(N_SIM, n))
            MKT_MAT[rid] = pair_mat(X["p_mkt"].values, GUM[rid], k)
        mm = pair_mat(X["p"].values, GUM[rid], k)
        iu = np.triu_indices(n, 1)
        pm, pk = mm[iu], MKT_MAT[rid][iu]
        keep = pm >= 0.10
        if not keep.any():
            continue
        nums = X["number"].astype(int).values
        for a_i, b_i, m_, k_ in zip(iu[0][keep], iu[1][keep], pm[keep], pk[keep]):
            a, b = sorted((nums[a_i], nums[b_i]))
            rows.append((rid, Y, m_, m_ / max(k_, 1e-9), WPAY.get((rid, f"{a}-{b}"), 0) / 100))
    W_ = pd.DataFrame(rows, columns=["race_id", "年", "pw", "ratio", "ret"])
    return T_, W_

In [ ]:
# ---------- 比較の部品（学習のやり方を選べるように拡張）----------
BASE_PRED = {}          # 今のモデルの予想（年・乱数ごと）。CatBoost との平均などで使い回す
CUR_Y = None
def _logit_mkt(d):
    p = d["p_mkt"].clip(1e-4, 0.999).values
    return np.log(p / (1 - p))
def _logp_mkt(d):
    return np.log(d["p_mkt"].clip(1e-6).values)
def _codes(d):
    return pd.factorize(d["race_id"])[0]
def _soft(codes, s):
    mx = np.full(codes.max() + 1, -np.inf)
    np.maximum.at(mx, codes, s)
    e = np.exp(s - mx[codes])
    return e / np.bincount(codes, weights=e)[codes]
def _soft_obj(codes):
    def f(y, s):
        p = _soft(codes, np.asarray(s, dtype=float))
        return p - np.asarray(y, dtype=float), np.maximum(p * (1 - p), 1e-6)
    return f
def _soft_metric(codes_by_len):
    def f(y, s):
        c = codes_by_len.get(len(y))
        if c is None:
            return "race_ll", 0.0, False
        p = _soft(c, np.asarray(s, dtype=float))
        return "race_ll", float(-np.log(np.clip(p[np.asarray(y) == 1], 1e-9, None)).mean()), False
    return f

def fit_predict(feats, mode, sd):
    # mode: "cls"（今のやり方）/ "rank" / dict(kind="cls"|"soft"|"cat", init=市場を出発点にするか, params=設定の上書き, decay=年の重み)
    o = {"kind": mode} if isinstance(mode, str) else dict(mode)
    P = {**base_params, "random_state": sd, **o.get("params", {})}
    sw = (o["decay"] ** (tr["year"].max() - tr["year"]).values) if o.get("decay") else None
    es = [lgb.early_stopping(100, verbose=False)]
    is_base = o == {"kind": "cls"} and list(feats) == list(features)
    if o["kind"] == "cat":
        if (CUR_Y, sd) not in BASE_PRED:
            fit_predict(features, "cls", sd)
        bf, be = BASE_PRED[(CUR_Y, sd)]
        from catboost import CatBoostClassifier
        cats = [c for c in feats if str(tr[c].dtype) == "category"]
        def cx(d):
            X = d[feats].copy()
            for c in cats:
                X[c] = X[c].astype(str)
            return X
        cb = CatBoostClassifier(iterations=1500, learning_rate=0.08, depth=6, random_seed=sd, verbose=0, thread_count=-1,
                                od_type="Iter", od_wait=100, allow_writing_files=False)
        cb.fit(cx(tr), tr["win"], cat_features=cats, eval_set=(cx(va_fit), va_fit["win"]), use_best_model=True)
        cf = race_norm(cb.predict_proba(cx(va_fit))[:, 1], va_fit["race_id"])
        ce = race_norm(cb.predict_proba(cx(va_ev))[:, 1], va_ev["race_id"])
        return (bf + cf) / 2, (be + ce) / 2, int(cb.get_best_iteration() or 0), cb
    if o["kind"] == "rank":
        m = lgb.LGBMRanker(**{**P, "objective": "lambdarank"})
        m.fit(tr[feats], tr["rel"], group=groups(tr), eval_set=[(va_fit[feats], va_fit["rel"])],
              eval_group=[groups(va_fit)], eval_at=[1], callbacks=es)
        pr = lambda d: race_softmax(m.predict(d[feats]), d["race_id"])
    elif o["kind"] == "soft":      # レース単位: 1レースの中で「どの馬が勝つか」を直接学習（レース内のソフトマックス）
        m = lgb.LGBMRegressor(**{**P, "objective": _soft_obj(_codes(tr)), "metric": "None"})
        kw = dict(init_score=_logp_mkt(tr), eval_init_score=[_logp_mkt(va_fit)]) if o.get("init") else {}
        m.fit(tr[feats], tr["win"].astype(float), sample_weight=sw, eval_set=[(va_fit[feats], va_fit["win"].astype(float))],
              eval_metric=_soft_metric({len(va_fit): _codes(va_fit)}), callbacks=es, **kw)
        pr = lambda d: race_softmax(m.predict(d[feats]) + (_logp_mkt(d) if o.get("init") else 0), d["race_id"])
    else:                          # 1頭ずつ「勝つか」を学習（今のやり方）。init=True なら市場の確率を出発点にして差だけを学習
        m = lgb.LGBMClassifier(**P)
        kw = dict(init_score=_logit_mkt(tr), eval_init_score=[_logit_mkt(va_fit)]) if o.get("init") else {}
        m.fit(tr[feats], tr["win"], sample_weight=sw, eval_set=[(va_fit[feats], va_fit["win"])], eval_metric="binary_logloss",
              callbacks=es, **kw)
        if o.get("init"):
            pr = lambda d: race_norm(1 / (1 + np.exp(-(m.predict(d[feats], raw_score=True) + _logit_mkt(d)))), d["race_id"])
        else:
            pr = lambda d: race_norm(m.predict_proba(d[feats])[:, 1], d["race_id"])
    pf, pe = pr(va_fit), pr(va_ev)
    if is_base:
        BASE_PRED[(CUR_Y, sd)] = (pf, pe)
    return pf, pe, m.best_iteration_, m

def compare(CMP):
    # CMP: {名前: (特徴量, 学習のやり方)}。先頭が比べる基準。並びが優先順（時間切れが近いと後ろから省略）
    global CUR_Y, ALL, TAN, WID, ES, NAMES, BASE_NAME
    ALL, TAN, WID, ES = {}, {k: [] for k in CMP}, {k: [] for k in CMP}, {k: [] for k in CMP}
    skipped = []
    for Y in YEARS:
        set_period(Y)
        CUR_Y = Y
        ALL[Y] = {}
        for name, (feats, mode) in CMP.items():
            if name in skipped:
                continue
            if (time.time() - T_NB) / 3600 > TIME_LIMIT_H and name != list(CMP)[0]:
                skipped.append(name)
                print(f"⚠ 時間切れが近いので「{name}」は省略します（結果は出しません）")
                continue
            try:
                r = run(f"{Y} {name}", feats, mode)
            except Exception as ex:
                skipped.append(name)
                print(f"⚠ 「{name}」はエラーのため省略: {type(ex).__name__}: {ex}")
                continue
            t_, w_ = collect(r["_E"], Y)
            TAN[name].append(t_); WID[name].append(w_)
            ES[name].append(r["_E"].assign(年=Y))
            r["_E"] = None
            ALL[Y][name] = r
        print(f"（ここまで {(time.time() - T_NB) / 3600:.1f}時間）")
    NAMES = [k for k in CMP if all(k in ALL[Y] for Y in YEARS)]
    BASE_NAME = list(CMP)[0]
    TAN = {k: pd.concat(TAN[k], ignore_index=True) for k in NAMES}
    WID = {k: pd.concat(WID[k], ignore_index=True) for k in NAMES}
    ES = {k: pd.concat(ES[k], ignore_index=True) for k in NAMES}
    if skipped:
        print("省略した案:", skipped)

def diffs(a, b, years=None):
    years = years or YEARS
    return np.concatenate([np.array(ALL[Y][a]["_seed"]) - np.array(ALL[Y][b]["_seed"]) for Y in years])

def judge():
    print("【年ごとの「市場との差」（かっこ内は基準との差。大きいほど良い）】")
    print(pd.DataFrame([{"年": Y, **{k: f"{ALL[Y][k]['市場との差']:.5f}（{ALL[Y][k]['市場との差'] - ALL[Y][BASE_NAME]['市場との差']:+.5f}）" for k in NAMES}}
                        for Y in YEARS]).set_index("年").T.to_string())
    print("\n【勝ち負け】（乱数10回×3年＝30回のうち、基準より良かった回数。24回以上で効果あり）")
    VERDICT.clear()
    for nm in NAMES[1:]:
        d = diffs(nm, BASE_NAME); w_ = int((d > 0).sum())
        v_ = "効果あり" if w_ >= 24 and d.mean() > 0 else ("逆効果" if w_ <= 6 else "誤差の範囲")
        VERDICT[nm] = (w_, v_, d.mean())
        print(f"▼ {nm}: {w_}勝{30 - w_}敗（{v_}）/ 平均の差 {d.mean():+.5f}（" +
              " / ".join(f"{Y}年 {int((diffs(nm, BASE_NAME, [Y]) > 0).sum())}勝" for Y in YEARS) + "）")
VERDICT = {}

# ---------- 回収率（本番の単勝＝◎・期待値1.05以上 を中心に、買い基準を広く振る）----------
FAV_EV = [1.0, 1.05, 1.1, 1.15, 1.2, 1.3]
EV_GRID, PT_GRID = [1.0, 1.1, 1.2, 1.3, 1.4, 1.5], [0.05, 0.10, 0.15, 0.20]
R_GRID, PW_GRID = [1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 2.0], [0.10, 0.15, 0.20, 0.25, 0.30]
def _fav(T_):
    return T_[T_["p"] == T_.groupby("race_id")["p"].transform("max")]
def _yr(s, col):
    return [s.loc[s["年"] == Y, col].mean() if (s["年"] == Y).any() else np.nan for Y in YEARS]
def roi_report():
    print("【単勝: ◎（1着確率1位）で期待値が基準以上なら買い】 3年合計（点数）/ 年ごと（" + " / ".join(map(str, YEARS)) + "）。★は本番の基準")
    rows = []
    for nm in NAMES:
        F_ = _fav(TAN[nm]).assign(ret=lambda x: x["odds"] * x["win"])
        row = {"案": nm}
        for ev in FAV_EV:
            s = F_[F_["ev"] >= ev]
            row[("★" if ev == 1.05 else "") + f"期待値{ev:.2f}"] = (f"{s['ret'].mean():.0%}（{len(s)}）/ " + " ".join(f"{y:.0%}" if y == y else "-" for y in _yr(s, "ret"))) if len(s) else "-"
        rows.append(row)
    print(pd.DataFrame(rows).set_index("案").T.to_string())
    # 広く振ったマス（1着率×期待値・1レース1頭）とワイド（度合い×確率・1レース1点）
    def tan_pick(T_, a, b):
        s = T_[(T_["ev"] >= a) & (T_["p"] >= b)]
        return s.loc[s.groupby("race_id")["ev"].idxmax()] if len(s) else s
    def wide_pick(W_, a, b):
        s = W_[(W_["ratio"] >= a) & (W_["pw"] >= b)]
        return s.loc[s.groupby("race_id")["ratio"].idxmax()] if len(s) else s
    G = {}
    for nm in NAMES:
        for kind in ["tan", "fav", "wide"]:
            out = {}
            if kind == "fav":
                F_ = _fav(TAN[nm])
                for a in FAV_EV:
                    s = F_[F_["ev"] >= a]; ret = s["odds"] * s["win"]
                    out[a] = (ret.mean() if len(s) else np.nan, len(s), [ret[s["年"] == Y].mean() if (s["年"] == Y).any() else np.nan for Y in YEARS])
            else:
                for a in (EV_GRID if kind == "tan" else R_GRID):
                    for b in (PT_GRID if kind == "tan" else PW_GRID):
                        s = tan_pick(TAN[nm], a, b) if kind == "tan" else wide_pick(WID[nm], a, b)
                        ret = s["odds"] * s["win"] if kind == "tan" else s["ret"]
                        out[(a, b)] = (ret.mean() if len(s) else np.nan, len(s), [ret[s["年"] == Y].mean() if (s["年"] == Y).any() else np.nan for Y in YEARS])
            G[(nm, kind)] = out
    rows = []
    for nm in NAMES:
        row = {"案": nm}
        for kind, lab in [("fav", "単勝◎"), ("tan", "単勝（印問わず）"), ("wide", "ワイド")]:
            cells = [c for c in G[(nm, kind)] if min(G[(nm, kind)][c][1], G[(BASE_NAME, kind)][c][1]) >= 100]
            if nm != BASE_NAME:
                win = sum(G[(nm, kind)][c][0] > G[(BASE_NAME, kind)][c][0] for c in cells)
                row[f"{lab} 基準を上回ったマス"] = f"{win}/{len(cells)}"
            row[f"{lab} 3年とも100%超"] = sum(all(y > 1 for y in G[(nm, kind)][c][2]) for c in G[(nm, kind)] if G[(nm, kind)][c][1] >= 90)
        rows.append(row)
    print("\n【買い基準を広く振ったときのマスの数】（1つの良いマスより、全体の傾向で判断）")
    print(pd.DataFrame(rows).set_index("案").to_string())
    print("\n【ワイド: 度合い1.6以上・ワイド確率0.2以上（本番の基準）】")
    for nm in NAMES:
        r, n, yr = G[(nm, "wide")][(1.6, 0.20)]
        print(f"  {nm}: 3年 {r:.0%}（{n}）/ " + " / ".join(f"{y:.0%}" if y == y else "-" for y in yr) if n else f"  {nm}: 該当なし")
    return G

# ---------- 一部の馬だけでの比較（どこで良くなったか）----------
def subset_report(masks, names=None):
    # 馬ごとの「勝つ・勝たない」の当てはまり（対数尤度）が、市場よりどれだけ良いか（1000頭あたり）。大きいほど良い
    names = names or NAMES
    rows = []
    for lab, f in masks.items():
        row = {"対象": lab}
        for nm in names:
            E_ = ES[nm]
            m_ = f(E_).fillna(False).astype(bool).values
            d = E_[m_]
            if len(d) < 300:
                continue
            y = d["win"].values
            def L(p):
                p = np.clip(p, 1e-6, 1 - 1e-6)
                return y * np.log(p) + (1 - y) * np.log(1 - p)
            g = (L(d["p"].values) - L(d["p_mkt"].values))
            row[nm] = f"{g.mean() * 1000:+.2f}"
            row["頭数"] = f"{len(d):,}"
        rows.append(row)
    t = pd.DataFrame(rows).set_index("対象")
    cols = ["頭数"] + [c for c in t.columns if c != "頭数"]
    print(t[cols].to_string())
    print("※ 1000頭あたりの「市場より当てはまりが良い量」。基準（左から2列目）より大きければ、その対象で良くなった")

def pick_report(masks, names=None):
    # 本番の単勝（◎・期待値1.05以上）の買い目を、対象ごとに分けた回収率（案ごと・3年合計）
    names = names or NAMES
    rows = []
    for lab, f in masks.items():
        row = {"対象": lab}
        for nm in names:
            E_ = ES[nm]
            F_ = E_[E_["p"] == E_.groupby("race_id")["p"].transform("max")]
            F_ = F_[(F_["ev"] >= 1.05) & f(F_).fillna(False).astype(bool)]
            ret = F_["odds"] * F_["win"]
            row[nm] = f"{ret.mean():.0%}（{len(F_)}）" if len(F_) else "-"
        rows.append(row)
    print(pd.DataFrame(rows).set_index("対象").to_string())
    print("※ 本番の単勝（◎・期待値1.05以上）のうち、その対象に当てはまる買い目だけの回収率（点数）。点数が少ない行は参考程度")

## ⑥ 比較（3年 × 乱数10回）

In [ ]:
COLS = list(dict.fromkeys(COLS + ["venue", "track_condition", "race_number", "distance"]))
CMP = {
    "今のモデル": (features, "cls"),
    "C1 市場を出発点": (features, dict(kind="cls", init=True)),
    "C2 レース単位": (features, dict(kind="soft")),
    "C1＋C2": (features, dict(kind="soft", init=True)),
    "C7 新しい年を重く": (features, dict(kind="cls", decay=0.85)),
    "C5 控えめ設定": (features, dict(kind="cls", params=dict(num_leaves=31, min_child_samples=300, colsample_bytree=0.6, reg_lambda=5.0))),
    "C5 大きめ設定": (features, dict(kind="cls", params=dict(num_leaves=127, min_child_samples=50))),
    "C6 CatBoostと平均": (features, dict(kind="cat")),
}
try:
    import catboost
except Exception:
    CMP.pop("C6 CatBoostと平均")
    print("⚠ CatBoost が使えないので C6 は省略します")
compare(CMP)
judge()

## ⑦ 回収率

In [ ]:
G = roi_report()

## ⑧ 学習し直さずに確率を直す（C3 オッズ帯の補正・C4 条件ごとの合成の重み）

各年とも、前半で補正の仕方を決めて**後半で採点**（今のモデルの予想＝乱数10回の平均を使う）。

In [ ]:
def wll(d, col):
    return -np.log(d.loc[d["win"] == 1, col].clip(1e-9)).mean()
def renorm(x, rid):
    x = pd.Series(np.asarray(x, dtype=float), index=rid.index)
    return (x / x.groupby(rid).transform("sum")).values
BANDS = [0, 3, 5, 10, 20, 50, 100, 1e9]
SEGS = {
    "競馬場": lambda d: d["venue"].astype(str),
    "頭数": lambda d: pd.cut(d["n_runners"], [0, 8, 10, 12, 99], labels=["〜8頭", "9〜10頭", "11〜12頭", "13頭〜"]).astype(str),
    "馬場": lambda d: np.where(d["track_condition"].astype(str).isin(["重", "不良"]), "道悪", "良・稍重"),
    "レース番号": lambda d: pd.cut(d["race_number"], [0, 4, 8, 10, 99], labels=["1〜4R", "5〜8R", "9〜10R", "11R〜"]).astype(str),
}
POST, OB = [], []
for Y in YEARS:
    set_period(Y)
    pf = np.mean([BASE_PRED[(Y, sd)][0] for sd in SEEDS], axis=0)
    pe = np.mean([BASE_PRED[(Y, sd)][1] for sd in SEEDS], axis=0)
    F = va_fit[COLS].assign(pm=pf).reset_index(drop=True)
    E = va_ev[COLS].assign(pm=pe).reset_index(drop=True)
    w = minimize(ll, x0=[0.5, 0.5], args=(F,), method="Nelder-Mead").x
    F["p"], E["p"] = comb(F, w).values, comb(E, w).values
    # C3: オッズ帯ごとの「実際÷予想」（少ない帯は1に寄せる）
    F["ob"], E["ob"] = pd.cut(F["odds"], BANDS), pd.cut(E["odds"], BANDS)
    gF = F.groupby("ob", observed=False)
    fac = (gF["win"].sum() + 10) / (gF["p"].sum() + 10)
    OB.append(fac.rename(Y))
    E["p_c3"] = renorm(E["p"] * E["ob"].map(fac).astype(float).values, E["race_id"])
    # C4: 区分ごとに合成の重みを決め直す（前半で300レース未満の区分は全体の重みのまま）
    for sname, f in SEGS.items():
        F["_s"], E["_s"] = f(F), f(E)
        out = np.zeros(len(E))
        for sv, Es in E.groupby("_s"):
            Fs = F[F["_s"] == sv]
            ws = minimize(ll, x0=w, args=(Fs,), method="Nelder-Mead").x if Fs["race_id"].nunique() >= 300 else w
            out[Es.index] = comb(Es, ws).values
        E[f"p_c4_{sname}"] = out
    E["年"] = Y
    POST.append(E)
POST = pd.concat(POST, ignore_index=True)
print("▼ C3 オッズ帯ごとの補正の倍率（前半で決めた値。1より大きい＝モデルが低く見ていた）")
print(pd.concat(OB, axis=1).round(3).to_string())
VARS = {"今のモデル": "p", "C3 オッズ帯の補正": "p_c3", **{f"C4 重みを{s}ごと": f"p_c4_{s}" for s in SEGS}}
rows = []
for lab, col in VARS.items():
    gy = [wll(POST[POST["年"] == Y], "p_mkt") - wll(POST[POST["年"] == Y], col) for Y in YEARS]
    T_ = POST.assign(_p=POST[col])
    F_ = T_[T_["_p"] == T_.groupby("race_id")["_p"].transform("max")]
    s = F_[F_["_p"] * F_["odds"] >= 1.05]
    ret = s["odds"] * s["win"]
    rows.append({"案": lab, "市場との差（年ごと）": " / ".join(f"{g:.5f}" for g in gy),
                 "今のモデルより良い年": "-" if col == "p" else f"{sum(g > b for g, b in zip(gy, base_gy))}/3",
                 "単勝◎1.05 3年": f"{ret.mean():.0%}（{len(s)}）",
                 "年ごと": " / ".join(f"{ret[s['年'] == Y].mean():.0%}" if (s["年"] == Y).any() else "-" for Y in YEARS)})
    if col == "p":
        base_gy = gy
print("\n▼ 確率を直した場合（年の並びは " + " / ".join(map(str, YEARS)) + "）")
print(pd.DataFrame(rows).to_string(index=False))
print("※ 3年とも良く、回収率も下がらなければ採用候補。学習し直しが要らないので、採用するときは予想用ノートブックに数行足すだけ")

## ⑨ D2 見送る条件（本番の単勝 ◎・期待値1.05以上 を区分ごとに）

In [ ]:
B_ = ES[BASE_NAME].copy()
B_ = B_[B_["p"] == B_.groupby("race_id")["p"].transform("max")]
B_ = B_[B_["ev"] >= 1.05].assign(ret=lambda x: x["odds"] * x["win"])
B_["競馬場"] = B_["venue"].astype(str)
B_["頭数"] = pd.cut(B_["n_runners"], [0, 8, 10, 12, 99], labels=["〜8頭", "9〜10頭", "11〜12頭", "13頭〜"]).astype(str)
B_["レース番号"] = pd.cut(B_["race_number"], [0, 4, 8, 10, 99], labels=["1〜4R", "5〜8R", "9〜10R", "11R〜"]).astype(str)
B_["馬場"] = B_["track_condition"].astype(str)
B_["距離"] = pd.cut(B_["distance"], [0, 1200, 1500, 1800, 9999], labels=["〜1200m", "1201〜1500m", "1501〜1800m", "1801m〜"]).astype(str)
B_["オッズ"] = pd.cut(B_["odds"], [0, 2, 3, 5, 10, 1e9], labels=["〜2倍", "2〜3倍", "3〜5倍", "5〜10倍", "10倍〜"]).astype(str)
FLAG = []
print(f"本番の単勝（今のモデル・3年分の後半）: {len(B_)}点 / 回収率 {B_['ret'].mean():.0%}")
for dim in ["競馬場", "頭数", "レース番号", "馬場", "距離", "オッズ"]:
    t = B_.groupby(dim).agg(点数=("ret", "size"), 的中率=("win", "mean"), 回収率=("ret", "mean"))
    for Y in YEARS:
        t[str(Y)] = B_[B_["年"] == Y].groupby(dim)["ret"].mean()
    print(f"\n【{dim}】")
    print(t.assign(的中率=lambda x: x["的中率"].map("{:.0%}".format), 回収率=lambda x: x["回収率"].map("{:.0%}".format),
                   **{str(Y): (lambda x, Y=Y: x[str(Y)].map(lambda v: "-" if v != v else f"{v:.0%}")) for Y in YEARS}).to_string())
    for k, r in t.iterrows():
        ys = [r[str(Y)] for Y in YEARS]
        if r["点数"] >= 60 and all(v == v for v in ys):
            if all(v < 0.85 for v in ys):
                FLAG.append((dim, k, int(r["点数"]), f"{r['回収率']:.0%}", "3年とも85%未満（見送り候補）"))
            elif all(v > 1.0 for v in ys):
                FLAG.append((dim, k, int(r["点数"]), f"{r['回収率']:.0%}", "3年とも100%超（得意な条件）"))
print("\n【3年とも同じ向きの区分】")
print(pd.DataFrame(FLAG, columns=["区分", "値", "点数", "3年合計", "向き"]).to_string(index=False) if FLAG else "  なし")
print("※ 点数が少ないので、区分を足し合わせると偶然が混ざる。見送るのは「3年とも85%未満」かつ理由が説明できるものだけにする")
print("\n【まとめ】")
for nm, (w_, v_, m_) in VERDICT.items():
    print(f"  {nm}: {w_}勝{30 - w_}敗 → {v_}")